In [35]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical
import torch.optim as optim

import numpy as np
import gymnasium as gym

In [3]:
torch.cuda.is_available()

True

In [4]:
env = gym.make("CartPole-v1", render_mode="rgb_array")

In [5]:
action_space_size = env.action_space.n
action_space_size

np.int64(2)

In [6]:
observation_space_size = env.observation_space.shape[0]
observation_space_size

4

In [9]:
env.observation_space.sample()

array([-4.346344  ,  0.7720449 ,  0.18694189, -1.8480256 ], dtype=float32)

In [49]:
env.reset()
env.step(env.action_space.sample())

(array([ 0.02209339, -0.17029716,  0.04185758,  0.3501325 ], dtype=float32),
 1.0,
 False,
 False,
 {})

In [7]:
import imageio

In [8]:
env.reset()
truncated = False
terminated = False
frames = []

while not (truncated or terminated):
    frames.append(env.render())
    state, reward, terminated, truncated, info = env.step(env.action_space.sample())

imageio.mimsave(
    "episode.mp4",
    frames,
    fps=30
)

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (600, 400) to (608, 400) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [42]:
class Policy(nn.Module):
    def __init__(self, action_space_size, hidden_size, observation_space_size):
        super().__init__()
        self.fc1 = nn.Linear(observation_space_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, action_space_size)

    def forward(self, inputs):
        out1 = F.relu(self.fc1(inputs))
        out2 = F.softmax(self.fc2(out1))
        return out2
    
    def act(self, state):
        state = torch.from_numpy(state).float().to(device)
        probs = self.forward(state)
        dist = Categorical(probs)
        action = dist.sample()
        return action.cpu().item(), dist.log_prob(action).cpu().item()

In [43]:
p = Policy(
    action_space_size=action_space_size,
    hidden_size=8,
    observation_space_size=observation_space_size
)

p = p.to(device)

In [44]:
sum(i.numel() for i in p.parameters())

58

In [45]:
p(torch.from_numpy(env.observation_space.sample()).float().to(device))

/tmp/ipykernel_17191/1418355859.py:9: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  out2 = F.softmax(self.fc2(out1))


tensor([0.6854, 0.3146], device='cuda:0', grad_fn=<SoftmaxBackward0>)

In [46]:
p.act(env.observation_space.sample())

/tmp/ipykernel_17191/1418355859.py:9: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  out2 = F.softmax(self.fc2(out1))


(0, -0.4332849979400635)

In [36]:
from collections import deque

In [90]:
def reinforce(
    policy: nn.Module,
    optimizer: optim.Optimizer,
    n_training_episodes: int,
    max_t: int,
    gamma: float,
    print_every: int
):
    scores_deque = deque(maxlen=100)
    scores = []

    for i_episode in range(1, n_training_episodes+1):
        saved_log_probs = []
        rewards = []

        state, _ = env.reset()
        for t in range(max_t):
            action, log_prob = policy.act(state)
            saved_log_probs.append(log_prob)
            state, reward, terminated, truncated, _ = env.step(action)
            rewards.append(reward)
            if terminated or truncated:
                break

        scores_deque.append(sum(rewards))
        scores.append(sum(rewards))

        returns = deque(maxlen=max_t)
        n_steps = len(rewards)

        for t in range(n_steps)[::-1]:
            prev = returns[0] if len(returns)>0 else 0
            returns.appendleft((prev * gamma) + rewards[t])

        eps = np.finfo(np.float32).eps.item()
        
        returns = torch.tensor(returns)
        returns = (returns - returns.mean()) / (returns.std() + eps)

        # print(f"returns at episode {i_episode}: {returns}")

        policy_loss = []
        for log_prob, disc_return in zip(saved_log_probs, returns):
            policy_loss.append(-log_prob * disc_return)

        print(f"policy loss at episode {i_episode}: {policy_loss}")
        policy_loss = torch.tensor(policy_loss).sum()

        optimizer.zero_grad()
        policy_loss.backward()
        optimizer.step()

        if i_episode%print_every == 0:
            print(f"Episode {i_episode}: \tAverage Score: {np.mean(scores_deque)}\tPolicy Loss: {policy_loss}\tDiscounted Reward: {returns[0]}")

    return scores

In [91]:
cartpole_hyperparameters = {
    "hidden_size": 16,
    "n_training_episodes": 1000,
    "n_eval_episodes": 10,
    "max_t": 1000,
    "gamma": 1.0,
    "lr": 1e-2,
    "env_id": "CartPole-v1",
    "state_space": observation_space_size,
    "action_space": action_space_size
}

In [92]:
cartpole_policy = Policy(
    action_space_size=cartpole_hyperparameters["action_space"],
    hidden_size=cartpole_hyperparameters["hidden_size"],
    observation_space_size=cartpole_hyperparameters["state_space"]
)

cartpole_policy = cartpole_policy.to(device)

In [93]:
cartpole_optimizer = optim.Adam(
    cartpole_policy.parameters(),
    lr=cartpole_hyperparameters["lr"]
)

In [94]:
scores = reinforce(
    policy=cartpole_policy,
    optimizer=cartpole_optimizer,
    n_training_episodes=cartpole_hyperparameters["n_training_episodes"],
    max_t=cartpole_hyperparameters['max_t'],
    gamma=cartpole_hyperparameters['gamma'],
    print_every=25
)

policy loss at episode 1: [tensor(1.3252), tensor(0.8139), tensor(0.9955), tensor(0.5808), tensor(0.4480), tensor(0.3266), tensor(0.3463), tensor(0.1094), tensor(0.), tensor(-0.1060), tensor(-0.2083), tensor(-0.3072), tensor(-0.4025), tensor(-0.4935), tensor(-1.1274), tensor(-1.2781), tensor(-0.8322)]


/tmp/ipykernel_17191/1418355859.py:9: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  out2 = F.softmax(self.fc2(out1))


RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [88]:
a = [torch.tensor(3.0), torch.tensor(4.0)]